In [19]:
# Download dataset
!wget https://archive.ics.uci.edu/ml/machine-learning-databases/00542/log2.csv

# Or use alternative URL
# !wget https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/Classification_evaluation_security_dataset_L2/log2.csv

--2026-06-17 11:24:48--  https://archive.ics.uci.edu/ml/machine-learning-databases/00542/log2.csv
Resolving archive.ics.uci.edu (archive.ics.uci.edu)... 128.195.10.252
Connecting to archive.ics.uci.edu (archive.ics.uci.edu)|128.195.10.252|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: unspecified
Saving to: ‘log2.csv.1’

log2.csv.1              [     <=>            ]   2.74M  2.38MB/s    in 1.2s    

2026-06-17 11:24:51 (2.38 MB/s) - ‘log2.csv.1’ saved [2876998]



In [20]:
!pip install scikit-plot imblearn

In [21]:
import scipy
import numpy as np
scipy.interp = np.interp
import pandas as pd
import os
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

# ---- PATCH for scikit-plot with newer scipy ----
import scipy
scipy.interp = np.interp
# ------------------------------------------------

import scikitplot as sk
from collections import Counter
from sklearn.model_selection import StratifiedKFold
from sklearn import *
import time
from imblearn.over_sampling import SMOTE

%matplotlib inline
sns.set(style="darkgrid")
plt.rcParams['figure.figsize'] = (15, 5)
plt.style.use('ggplot')
seed = 42

import warnings
warnings.filterwarnings(action="ignore", category=FutureWarning)
!pip install scikit-plot imblearn

In [22]:
df = pd.read_csv('log2.csv', header=0)
df.head()

,Source Port,Destination Port,NAT Source Port,NAT Destination Port,Action,Bytes,Bytes Sent,Bytes Received,Packets,Elapsed Time (sec),pkts_sent,pkts_received
0,57222,53,54587,53,allow,177,94,83,2,30,1,1
1,56258,3389,56258,3389,allow,4768,1600,3168,19,17,10,9
2,6881,50321,43265,50321,allow,238,118,120,2,1199,1,1
3,50553,3389,50553,3389,allow,3327,1438,1889,15,17,8,7
4,50002,443,45848,443,allow,25358,6778,18580,31,16,13,18


In [27]:
df.shape                     # (65532, 12)
df.columns
df.info()
df.describe().T

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 65532 entries, 0 to 65531
Data columns (total 12 columns):
 #   Column                Non-Null Count  Dtype 
---  ------                --------------  ----- 
 0   Source Port           65532 non-null  int64 
 1   Destination Port      65532 non-null  int64 
 2   NAT Source Port       65532 non-null  int64 
 3   NAT Destination Port  65532 non-null  int64 
 4   Action                65532 non-null  object
 5   Bytes                 65532 non-null  int64 
 6   Bytes Sent            65532 non-null  int64 
 7   Bytes Received        65532 non-null  int64 
 8   Packets               65532 non-null  int64 
 9   Elapsed Time (sec)    65532 non-null  int64 
 10  pkts_sent             65532 non-null  int64 
 11  pkts_received         65532 non-null  int64 
dtypes: int64(11), object(1)
memory usage: 6.0+ MB


,count,mean,std,min,25%,50%,75%,max
Source Port,65532.0,49391.969343,1.525571e+04,0.0,49183.0,53776.5,58638.00,6.553400e+04
Destination Port,65532.0,10577.385812,1.846603e+04,0.0,80.0,445.0,15000.00,6.553500e+04
NAT Source Port,65532.0,19282.972761,2.197069e+04,0.0,0.0,8820.5,38366.25,6.553500e+04
NAT Destination Port,65532.0,2671.049930,9.739162e+03,0.0,0.0,53.0,443.00,6.553500e+04
Bytes,65532.0,97123.950085,5.618439e+06,60.0,66.0,168.0,752.25,1.269359e+09
Bytes Sent,65532.0,22385.796908,3.828139e+06,60.0,66.0,90.0,210.00,9.484772e+08
Bytes Received,65532.0,74738.153177,2.463208e+06,0.0,0.0,79.0,449.00,3.208818e+08
Packets,65532.0,102.866035,5.133002e+03,1.0,1.0,2.0,6.00,1.036116e+06
Elapsed Time (sec),65532.0,65.833577,3.024618e+02,0.0,0.0,15.0,30.00,1.082400e+04
pkts_sent,65532.0,41.399530,3.218871e+03,1.0,1.0,1.0,3.00,7.475200e+05


In [ ]:
AUTOTUNE = os.cpu_count()
Classifiers = {
    "Linear_Regressor": linear_model.LogisticRegression(C=10, solver='liblinear'),
    "Random_Forest": ensemble.RandomForestClassifier(random_state=seed, n_jobs=AUTOTUNE),
    "Ada_Boost": ensemble.AdaBoostClassifier(tree.DecisionTreeClassifier(random_state=seed),
                                             random_state=seed, learning_rate=0.1),
    "Ex_Tree_Classifier": ensemble.ExtraTreesClassifier(random_state=seed, n_jobs=AUTOTUNE),
    "kNN": neighbors.KNeighborsClassifier(n_jobs=AUTOTUNE),
    "Decision_Tree": tree.DecisionTreeClassifier(random_state=seed)
}
start_time = time.time()
kfold = model_selection.StratifiedKFold(n_splits=10)
for name, classifier in Classifiers.items():
    sk.estimators.plot_learning_curve(classifier, xtrain, ytrain,
                                      cv=kfold, title=name,
                                      random_state=seed, n_jobs=-1)
print("Elapsed time:", round(time.time()-start_time, 2), 'sec')

In [ ]:
logging_cols = ["Classifier", "Test accuracy", "Train accuracy", "Log_loss", "Correct_cases", "Incorrect_cases"]
logging = pd.DataFrame(columns=logging_cols)

start_time = time.time()
for name_c, classif_c in Classifiers.items():
    classif_c.fit(xtrain, ytrain)
    predict = classif_c.predict(xtest)
    accuracy = classif_c.score(xtest, ytest)
    train_pred = classif_c.predict_proba(xtest)
    lg_ls = log_loss(ytest, train_pred)
    train_test_acc = classif_c.score(xtrain, ytrain)
    correct = (ytest == predict).sum()
    incorrect = (ytest != predict).sum()

    print("="*45)
    print(f'**** Estimations for {name_c} ****')
    print(f"Test accuracy:  {accuracy*100:.2f}%")
    print(f"Train accuracy: {train_test_acc*100:.2f}%")
    print(f"Log_loss: {lg_ls:.4f}")
    print(f"Correct points: {correct}, Incorrect points: {incorrect}")
    sk.metrics.plot_confusion_matrix(ytest, predict, title=name_c, cmap=plt.cm.Greens)

    logging_entry = pd.DataFrame([[name_c, accuracy*100, train_test_acc*100, lg_ls, correct, incorrect]],
                                  columns=logging_cols)
    logging = pd.concat([logging, logging_entry], ignore_index=True)
print("\nElapsed time:", round(time.time()-start_time, 2), 'sec')

In [ ]:
sns.set_color_codes("muted")
graph = sns.barplot(x='Classifier', y='Test accuracy', data=logging, palette='hls')
for p in graph.patches:
    graph.annotate(format(p.get_height(), '.2f'),
                   (p.get_x() + p.get_width()/2., p.get_height()),
                   ha='center', va='center', xytext=(0, -12), size=12,
                   textcoords='offset points')
plt.xlabel('\nClassifier name')
plt.title('Test Accuracy of Classifiers')
plt.show()
sns.set_color_codes("muted")
graph = sns.barplot(x='Classifier', y='Log_loss', data=logging, palette='hls')
for p in graph.patches:
    graph.annotate(format(p.get_height(), '.4f'),
                   (p.get_x() + p.get_width()/2., p.get_height()),
                   ha='center', va='center', xytext=(0, 5), size=12,
                   textcoords='offset points')
plt.xlabel('\nClassifier name')
plt.title('Log Loss (lower is better)')
plt.show()
sns.set_color_codes("muted")
graph = sns.barplot(x='Classifier', y='Correct_cases', data=logging, palette='hls')
for p in graph.patches:
    graph.annotate(format(p.get_height(), '.0f'),
                   (p.get_x() + p.get_width()/2., p.get_height()),
                   ha='center', va='center', xytext=(0, -12), size=12,
                   textcoords='offset points')
plt.xlabel('\nClassifier name')
plt.title('Number of Correct Predictions')
plt.show()
sns.set_color_codes("muted")
graph = sns.barplot(x='Classifier', y='Incorrect_cases', data=logging, palette='hls')
for p in graph.patches:
    graph.annotate(format(p.get_height(), '.0f'),
                   (p.get_x() + p.get_width()/2., p.get_height()),
                   ha='center', va='center', xytext=(0, 5), size=12,
                   textcoords='offset points')
plt.xlabel('\nClassifier name')
plt.title('Number of Incorrect Predictions')
plt.show()
